# Discourse Token Eigenspectrum Analysis

This notebook tests whether discourse-token hidden-state geometry distinguishes correct and incorrect reasoning traces.

Implemented metrics per trace/layer:

- `disc_eff_rank`: effective rank of discourse-token covariance spectrum
- `disc_top_ratio`: top eigenvalue ratio (`lambda_1 / sum(lambda_i)`)
- `disc_rayleigh_local_*`: local (within-trace) Rayleigh summaries
- `disc_rayleigh_global_*`: Rayleigh summaries against global discourse covariance

Then we evaluate AUC with leakage-safe CV and compare:

- entropy-only baseline
- discourse-only features
- entropy + discourse features
- Mahalanobis-only and entropy+Mahalanobis baselines
- entropy + Mahalanobis + discourse features


In [1]:
# Setup
from __future__ import annotations

import os
import re
import sys
from pathlib import Path

import numpy as np
from sklearn.model_selection import StratifiedKFold

candidate_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "analyze.py").exists() and (candidate / "data").exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate repository root containing analyze.py and data/")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analyze import (
    _fold_clf_auc,
    build_feature_matrix,
    compute_mahal_distances,
    detect_layers,
    entropy_features,
    evaluate_features,
    fit_mahalanobis_reference_safe,
    load_all_traces,
    mahal_features,
)

SEED = 42
DATA_DIR = os.environ.get("TRACE_DATA_DIR", str(REPO_ROOT / "data" / "deepseek" / "math500"))
LAYERS = None  # e.g. [2, 7, 14, 21]; None => auto-detect
N_SPLITS = 5
PCA_DIM = 128

# Marker set is intentionally broad; adjust per model/tokenizer behavior.
DISCOURSE_MARKERS = {
    "wait", "hmm", "therefore", "alternatively", "however",
    "but", "so", "thus", "actually", "reconsider", "instead",
}

# Fallback when token text is unavailable in saved traces.
FALLBACK_ENTROPY_QUANTILE = 0.85
MIN_DISCOURSE_TOKENS = 3

print(f"seed={SEED} | DATA_DIR={DATA_DIR}")


seed=42 | DATA_DIR=/home/djaniak/projects/reasoning-geometry-probe/data/deepseek/math500


## Data Loading Notes

Current collector files in this repo store hidden states and entropy, but not guaranteed token text per generated token.

This notebook supports two position-selection modes:

1. **Marker mode** (preferred): use saved token strings when present.
2. **Entropy proxy mode** (fallback): use high-entropy token positions as discourse proxies.

If you want strict discourse-marker analysis, add token strings to collection output and rerun this notebook in marker mode.


In [2]:
# Loading + discourse position helpers

def _normalize_token(tok: str) -> str:
    tok = tok.lower().strip()
    tok = tok.replace("▁", "").replace("Ġ", "")
    tok = re.sub(r"^[^a-z0-9]+|[^a-z0-9]+$", "", tok)
    return tok


def is_discourse_token(tok: str, markers: set[str]) -> bool:
    nt = _normalize_token(tok)
    if not nt:
        return False
    if nt in markers:
        return True
    # Handle simple inflections/pieces like "therefore," or "wait..."
    return any(nt.startswith(m) for m in markers)


def get_discourse_positions(trace: dict, mode: str = "auto") -> list[int]:
    """
    mode:
      - 'markers': use token strings only
      - 'entropy_proxy': use high-entropy positions only
      - 'auto': markers if available, otherwise entropy proxy
    """
    tokens = trace.get("tokens")

    if mode in ("markers", "auto") and tokens is not None:
        pos = [i for i, tok in enumerate(tokens) if is_discourse_token(str(tok), DISCOURSE_MARKERS)]
        if mode == "markers" or pos:
            return pos

    if mode in ("entropy_proxy", "auto"):
        e = trace["entropies"]
        thr = np.quantile(e, FALLBACK_ENTROPY_QUANTILE)
        return np.where(e >= thr)[0].tolist()

    return []


def to_eigenvalues_from_tokens(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return (eigvals, mean_vec, V) using SVD in token x hidden space."""
    if X.ndim != 2 or len(X) < 2:
        return np.array([]), np.array([]), np.array([[]])

    mu = X.mean(axis=0)
    Xc = X - mu
    # Xc = U S V^T, covariance eigenvalues = S^2 / (n-1)
    _, s, vt = np.linalg.svd(Xc, full_matrices=False)
    eigvals = (s ** 2) / max(len(X) - 1, 1)
    V = vt.T
    return eigvals, mu, V


def effective_rank(eigvals: np.ndarray, eps: float = 1e-12) -> float:
    if eigvals.size == 0:
        return np.nan
    lam = eigvals[eigvals > eps]
    if lam.size == 0:
        return np.nan
    p = lam / lam.sum()
    return float(np.exp(-(p * np.log(p + eps)).sum()))


def top_eigen_ratio(eigvals: np.ndarray, eps: float = 1e-12) -> float:
    if eigvals.size == 0:
        return np.nan
    denom = float(eigvals.sum())
    if denom <= eps:
        return np.nan
    return float(eigvals.max() / denom)


def rayleigh_scores(X: np.ndarray, eigvals: np.ndarray, mu: np.ndarray, V: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Regularized Rayleigh in covariance eigenbasis for each row in X."""
    if X.size == 0 or eigvals.size == 0 or V.size == 0:
        return np.array([])

    Xc = X - mu
    proj = Xc @ V
    numer = (proj ** 2 / (eigvals + eps)).sum(axis=1)
    denom = np.maximum((Xc ** 2).sum(axis=1), eps)
    return numer / denom


def discourse_metrics_for_trace(trace: dict, layer: int, global_model: dict | None = None, mode: str = "auto") -> dict:
    pos = get_discourse_positions(trace, mode=mode)
    out = {
        "n_discourse": len(pos),
        "disc_eff_rank": np.nan,
        "disc_top_ratio": np.nan,
        "disc_rayleigh_local_mean": np.nan,
        "disc_rayleigh_local_max": np.nan,
        "disc_rayleigh_global_mean": np.nan,
        "disc_rayleigh_global_max": np.nan,
        "usable": 0,
    }

    if len(pos) < MIN_DISCOURSE_TOKENS:
        return out

    X = trace["hiddens"][layer][pos]
    eigvals, mu, V = to_eigenvalues_from_tokens(X)
    r_local = rayleigh_scores(X, eigvals, mu, V)

    out["disc_eff_rank"] = effective_rank(eigvals)
    out["disc_top_ratio"] = top_eigen_ratio(eigvals)
    out["disc_rayleigh_local_mean"] = float(np.mean(r_local)) if r_local.size else np.nan
    out["disc_rayleigh_local_max"] = float(np.max(r_local)) if r_local.size else np.nan

    if global_model is not None:
        r_global = rayleigh_scores(X, global_model["eigvals"], global_model["mu"], global_model["V"])
        out["disc_rayleigh_global_mean"] = float(np.mean(r_global)) if r_global.size else np.nan
        out["disc_rayleigh_global_max"] = float(np.max(r_global)) if r_global.size else np.nan

    out["usable"] = 1
    return out


# Optional token attachment + leakage-safe feature extraction


In [3]:
def attach_optional_tokens(traces: list[dict], data_dir: str) -> None:
    """Best-effort attach per-token strings as trace['tokens']."""
    by_trace_id = {t["trace_id"]: t for t in traces}
    by_idx = {t["idx"]: t for t in traces}

    for fname in sorted(os.listdir(data_dir)):
        if not fname.endswith(".npz"):
            continue
        data = np.load(os.path.join(data_dir, fname), allow_pickle=True)
        for key in data.files:
            if not (key.startswith("tokens_") or key.startswith("token_text_") or key.startswith("generated_tokens_")):
                continue

            m = re.search(r"(\d+)$", key)
            if m is None:
                continue
            sid = int(m.group(1))

            arr = data[key]
            tok = [str(x) for x in arr.tolist()]
            if sid in by_trace_id:
                by_trace_id[sid]["tokens"] = tok
            elif sid in by_idx:
                by_idx[sid]["tokens"] = tok


def build_global_discourse_model(traces: list[dict], layer: int, mode: str = "auto") -> dict | None:
    mats = []
    for t in traces:
        pos = get_discourse_positions(t, mode=mode)
        if len(pos) < MIN_DISCOURSE_TOKENS:
            continue
        mats.append(t["hiddens"][layer][pos])

    if not mats:
        return None

    X = np.concatenate(mats, axis=0)
    eigvals, mu, V = to_eigenvalues_from_tokens(X)
    return {"eigvals": eigvals, "mu": mu, "V": V, "n_tokens": len(X)}


def discourse_feature_matrix_with_model(
    traces: list[dict],
    layer: int,
    global_model: dict | None,
    mode: str = "auto",
) -> tuple[np.ndarray, np.ndarray]:
    rows = []
    usable = []
    for t in traces:
        m = discourse_metrics_for_trace(t, layer, global_model=global_model, mode=mode)
        rows.append([
            m["n_discourse"],
            m["disc_eff_rank"],
            m["disc_top_ratio"],
            m["disc_rayleigh_local_mean"],
            m["disc_rayleigh_local_max"],
            m["disc_rayleigh_global_mean"],
            m["disc_rayleigh_global_max"],
        ])
        usable.append(m["usable"])

    X = np.array(rows, dtype=float)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X, np.array(usable, dtype=int)


def evaluate_discourse_foldwise(
    traces: list[dict],
    layer: int,
    y: np.ndarray,
    fold_indices: list[tuple[np.ndarray, np.ndarray]],
    X_ent: np.ndarray,
    mode: str = "auto",
) -> tuple[dict, dict, float]:
    roc_d, pr_d = [], []
    roc_ed, pr_ed = [], []
    usable_all = []

    for train_idx, test_idx in fold_indices:
        g = build_global_discourse_model([traces[i] for i in train_idx], layer, mode=mode)
        X_d, usable = discourse_feature_matrix_with_model(traces, layer, g, mode=mode)
        usable_all.append(float(usable.mean()))

        out_d = _fold_clf_auc(X_d, y, train_idx, test_idx)
        if out_d is not None:
            roc_d.append(out_d[0]); pr_d.append(out_d[1])

        X_ed = np.hstack([X_ent, X_d])
        out_ed = _fold_clf_auc(X_ed, y, train_idx, test_idx)
        if out_ed is not None:
            roc_ed.append(out_ed[0]); pr_ed.append(out_ed[1])

    disc = {
        "roc_auc_mean": float(np.mean(roc_d)) if roc_d else np.nan,
        "roc_auc_std": float(np.std(roc_d)) if roc_d else np.nan,
        "pr_auc_mean": float(np.mean(pr_d)) if pr_d else np.nan,
        "pr_auc_std": float(np.std(pr_d)) if pr_d else np.nan,
    }
    ent_disc = {
        "roc_auc_mean": float(np.mean(roc_ed)) if roc_ed else np.nan,
        "roc_auc_std": float(np.std(roc_ed)) if roc_ed else np.nan,
        "pr_auc_mean": float(np.mean(pr_ed)) if pr_ed else np.nan,
        "pr_auc_std": float(np.std(pr_ed)) if pr_ed else np.nan,
    }
    return disc, ent_disc, float(np.mean(usable_all)) if usable_all else 0.0


def evaluate_entropy_mahal_discourse_combo(
    traces: list[dict],
    layer: int,
    y: np.ndarray,
    fold_indices: list[tuple[np.ndarray, np.ndarray]],
    X_ent: np.ndarray,
    mode: str = "auto",
) -> dict:
    """Leakage-safe fold-wise evaluation of entropy + Mahalanobis + discourse."""
    roc_combo, pr_combo = [], []

    for train_idx, test_idx in fold_indices:
        correct_train = [traces[i] for i in train_idx if traces[i]["is_correct"]]
        ref = fit_mahalanobis_reference_safe(correct_train, layer, PCA_DIM)
        if ref is None:
            continue
        pca, mu, cov_inv = ref

        g = build_global_discourse_model([traces[i] for i in train_idx], layer, mode=mode)
        X_disc, _ = discourse_feature_matrix_with_model(traces, layer, g, mode=mode)

        rows = []
        for i, trace in enumerate(traces):
            e = trace["entropies"]
            m = compute_mahal_distances(trace["hiddens"][layer], pca, mu, cov_inv)
            rows.append(X_ent[i].tolist() + mahal_features(e, m) + X_disc[i].tolist())
        X_all = np.array(rows)

        out = _fold_clf_auc(X_all, y, train_idx, test_idx)
        if out is not None:
            roc_combo.append(out[0]); pr_combo.append(out[1])

    return {
        "roc_auc_mean": float(np.mean(roc_combo)) if roc_combo else np.nan,
        "roc_auc_std": float(np.std(roc_combo)) if roc_combo else np.nan,
        "pr_auc_mean": float(np.mean(pr_combo)) if pr_combo else np.nan,
        "pr_auc_std": float(np.std(pr_combo)) if pr_combo else np.nan,
    }


DISCOURSE_FEATURE_NAMES = [
    "n_discourse",
    "disc_eff_rank",
    "disc_top_ratio",
    "disc_rayleigh_local_mean",
    "disc_rayleigh_local_max",
    "disc_rayleigh_global_mean",
    "disc_rayleigh_global_max",
]

In [ ]:
# Run experiment across layers
if LAYERS is None:
    LAYERS = detect_layers(DATA_DIR)

traces = load_all_traces(DATA_DIR, LAYERS)
attach_optional_tokens(traces, DATA_DIR)

X_ent, y = build_feature_matrix(traces)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_indices = list(skf.split(X_ent, y))

entropy_baseline = evaluate_features(X_ent, y, fold_indices, "entropy_only")

results = {
    "entropy_only": entropy_baseline,
    "layers": {},
    "n_traces": len(traces),
    "n_correct": int(y.sum()),
    "n_incorrect": int((1 - y).sum()),
}

for layer in LAYERS:
    print("\n" + "=" * 60)
    print(f"Layer {layer}")

    disc_only, ent_plus_disc, usable_frac = evaluate_discourse_foldwise(
        traces, layer, y, fold_indices, X_ent, mode="auto"
    )

    # Existing Mahalanobis baselines from fold-wise train-only references.
    from analyze import evaluate_foldwise_mahalanobis
    mah = evaluate_foldwise_mahalanobis(
        traces, layer, PCA_DIM, y, fold_indices, label_prefix=f"L{layer} "
    )

    ent_mah_disc = evaluate_entropy_mahal_discourse_combo(
        traces, layer, y, fold_indices, X_ent, mode="auto"
    )

    results["layers"][layer] = {
        "discourse_feature_names": DISCOURSE_FEATURE_NAMES,
        "usable_fraction": usable_frac,
        "discourse_only": disc_only,
        "entropy_plus_discourse": ent_plus_disc,
        "mahalanobis_only": mah["mahalanobis_only"],
        "entropy_plus_mahalanobis": mah["combined"],
        "entropy_plus_mahalanobis_plus_discourse": ent_mah_disc,
        "delta_disc_vs_entropy": float(disc_only["roc_auc_mean"] - entropy_baseline["roc_auc_mean"]),
        "delta_ent_disc_vs_entropy": float(ent_plus_disc["roc_auc_mean"] - entropy_baseline["roc_auc_mean"]),
        "delta_ent_mah_disc_vs_ent_mah": float(
            ent_mah_disc["roc_auc_mean"] - mah["combined"]["roc_auc_mean"]
        ),
    }

results


    entropy_only                  : ROC-AUC = 0.7755 ± 0.0537 | PR-AUC = 0.7085 ± 0.0549

Layer 7
  Layer 7: PCA var=0.551, cov cond=26.6, fit on 231215 tokens
  Layer 7: PCA var=0.551, cov cond=27.0, fit on 236177 tokens
  Layer 7: PCA var=0.553, cov cond=26.8, fit on 229371 tokens
  Layer 7: PCA var=0.553, cov cond=26.7, fit on 228019 tokens
  Layer 7: PCA var=0.554, cov cond=26.7, fit on 233594 tokens
    L7 mahalanobis_only              : ROC-AUC = 0.8261 ± 0.0222 | PR-AUC = 0.7344 ± 0.0404
    L7 combined                      : ROC-AUC = 0.8589 ± 0.0183 | PR-AUC = 0.7763 ± 0.0359
    L7 combined+length               : ROC-AUC = 0.8703 ± 0.0206 | PR-AUC = 0.7839 ± 0.0353
  Layer 7: PCA var=0.551, cov cond=26.6, fit on 231215 tokens
  Layer 7: PCA var=0.551, cov cond=27.0, fit on 236177 tokens
  Layer 7: PCA var=0.553, cov cond=26.8, fit on 229371 tokens
  Layer 7: PCA var=0.553, cov cond=26.7, fit on 228019 tokens
  Layer 7: PCA var=0.554, cov cond=26.7, fit on 233594 tokens

Layer

# Compact summary table

In [ ]:
print(f"Traces: {results['n_traces']} (correct={results['n_correct']}, incorrect={results['n_incorrect']})")
print(f"Entropy baseline ROC-AUC: {results['entropy_only']['roc_auc_mean']:.4f}")
print()
print(
    f"{'layer':>6} {'usable':>8} {'disc':>8} {'ent+disc':>10} "
    f"{'mah':>8} {'ent+mah':>10} {'ent+mah+disc':>14} {'d(ent+mah)':>11}"
)
for layer in sorted(results['layers']):
    r = results['layers'][layer]
    print(
        f"{layer:>6} "
        f"{r['usable_fraction']:>8.2f} "
        f"{r['discourse_only']['roc_auc_mean']:>8.4f} "
        f"{r['entropy_plus_discourse']['roc_auc_mean']:>10.4f} "
        f"{r['mahalanobis_only']['roc_auc_mean']:>8.4f} "
        f"{r['entropy_plus_mahalanobis']['roc_auc_mean']:>10.4f} "
        f"{r['entropy_plus_mahalanobis_plus_discourse']['roc_auc_mean']:>14.4f} "
        f"{r['delta_ent_mah_disc_vs_ent_mah']:>11.4f}"
    )

best = max(
    results['layers'].items(),
    key=lambda kv: kv[1]['entropy_plus_mahalanobis_plus_discourse']['roc_auc_mean'],
)
print(f"\nBest layer for ent+mah+disc: L{best[0]} @ {best[1]['entropy_plus_mahalanobis_plus_discourse']['roc_auc_mean']:.4f}")
